In [0]:
print("Hello World!...Week04 case study I'm doing.")

Hello World!...Week04 case study I'm doing.


In [0]:
%pip install great_expectations==0.15.39
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
sas_token = "sp=racwdlme&st=2025-08-08T09:28:07Z&se=2025-08-08T17:43:07Z&spr=https&sv=2024-11-04&sr=c&sig=uOFSJEnbS0ZFkIh%2BIhbCis%2FsmUECUyFono1G4OVIl8Q%3D"
storage_account = "storage2madhu"
container = "output-container"
delta_path = f"wasbs://{container}@{storage_account}.blob.core.windows.net/raw/bronze/"
spark.conf.set(f"fs.azure.sas.{container}.{storage_account}.blob.core.windows.net", sas_token)

tables = [
    "policy", "policyholder", "coverage", "premium", "claims",
    "beneficiaries", "agents", "policydocuments", "policystatus", "payments"
]

table_dfs = {}

for table in tables:
    path = delta_path + table
    df = spark.read.format("parquet").load(path)
    table_dfs[table] = df
    # display(df)



In [0]:
import great_expectations as ge
from great_expectations.dataset import SparkDFDataset

def infer_keys(df):
    keys = []
    for col in df.columns:
        if col.endswith("_id") or col.endswith("_ID") or col.endswith("_Id") or col.endswith("_ID") or col.endswith("id") or col.endswith("ID") or col.endswith("_Id"):
            keys.append(col)
    return keys
validation_result = {}
for table_name, df in table_dfs.items():
    print(f"\nValidating table: {table_name}, columns: {df.columns}")
    ge_df = SparkDFDataset(df)

    # ge_df.expect_table_row_count_to_be_greater_than(0)

    key_columns = infer_keys(df)
    print(f"Inferred keys: {key_columns}")

    if not key_columns:
        print(f"No key columns found for {table_name}, skipping key checks.")
    else:
        for key in key_columns:
            print(f"Checking key column: {key}")
            ge_df.expect_column_to_exist(key)
            ge_df.expect_column_values_to_not_be_null(key)
            ge_df.expect_column_values_to_be_unique(key)

    result = ge_df.validate()
    validation_result[table_name] = result
    # print("Validation result summary:", result)
    if result["success"]:
        print(f"{table_name} Passed validation")
        df.spark.write.mode("overwrite").option("header", "true").saveAsTable")
    else:
        print(f"{table_name} Failed validation")

    print(validation_result)
# ge.data_context.DataContext().save_expectation_suite(expectation_suite_name="my_expectation_suite")



Validating table: policy, columns: ['PolicyID', 'PolicyNumber', 'PolicyType', 'StartDate', 'EndDate', 'PolicyHolderID', 'StatusID']
Inferred keys: ['PolicyID', 'PolicyHolderID', 'StatusID']
Checking key column: PolicyID
Checking key column: PolicyHolderID
Checking key column: StatusID
policy Failed validation
{'policy': {
  "success": false,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "expectation_type": "expect_column_to_exist",
        "kwargs": {
          "column": "PolicyID",
          "result_format": "BASIC"
        },
        "meta": {}
      },
      "result": {},
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_message": null,
        "exception_traceback": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "expectation_type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "column": "PolicyID",
          "result_format": "BASIC"
   

In [0]:
from pyspark.sql.functions import regexp_replace, lit, concat, substring

sas_token = "sp=racwdlme&st=2025-08-08T09:28:07Z&se=2025-08-08T17:43:07Z&spr=https&sv=2024-11-04&sr=c&sig=uOFSJEnbS0ZFkIh%2BIhbCis%2FsmUECUyFono1G4OVIl8Q%3D"
storage_account = "storage2madhu"
container = "output-container"
silver_data_path = f"wasbs://{container}@{storage_account}.blob.core.windows.net/raw/silver/"
gold_data_path = f"wasbs://{container}@{storage_account}.blob.core.windows.net/raw/gold/"
spark.conf.set(f"fs.azure.sas.{container}.{storage_account}.blob.core.windows.net", sas_token)

# Masking functions for each table
def mask_policy(df):
    return df

def mask_policyholder(df):
    return df.withColumn("FirstName", regexp_replace("FirstName", ".", "*")) \
             .withColumn("LastName", regexp_replace("LastName", ".", "*")) \
             .withColumn("DateOfBirth", lit("1900-01-01")) \
             .withColumn("Email", regexp_replace("Email", r"@.*", "@***")) \
             .withColumn("Phone", concat(lit("XXX-XXX-"), substring("Phone", -4, 4))) \
             .withColumn("Address", lit("************"))

def mask_coverage(df):
    return df

def mask_premium(df):
    return df

def mask_claims(df):
    return df

def mask_beneficiaries(df):
    return df.withColumn("FirstName", regexp_replace("FirstName", ".", "*")) \
             .withColumn("LastName", regexp_replace("LastName", ".", "*"))

def mask_agents(df):
    return df.withColumn("FirstName", regexp_replace("FirstName", ".", "*")) \
             .withColumn("LastName", regexp_replace("LastName", ".", "*")) \
             .withColumn("Email", regexp_replace("Email", r"@.*", "****@")) \
             .withColumn("Phone", concat(lit("XXX-XXX-"), substring("Phone", -4, 4)))

def mask_policydocuments(df):
    return df.withColumn("DocumentURL", lit("*****masked*****"))

def mask_policystatus(df):
    return df

def mask_payments(df):
    return df.withColumn("PaymentMethod", lit("masked"))

mask_functions = {
    "policy": mask_policy,
    "policyholder": mask_policyholder,
    "coverage": mask_coverage,
    "premium": mask_premium,
    "claims": mask_claims,
    "beneficiaries": mask_beneficiaries,
    "agents": mask_agents,
    "policydocuments": mask_policydocuments,
    "policystatus": mask_policystatus,
    "payments": mask_payments,
}

for table_name, df in table_dfs.items():
    print(f"Processing table: {table_name}")

    mask_fn = mask_functions.get(table_name.lower(), lambda x: x)
    masked_df = mask_fn(df)

    # Show sample of masked data
    masked_df.show(5)

    output_path = f"{silver_data_path}/{table_name}"

    try:
        # Save masked data as parquet for better efficiency and schema
        masked_df.write.mode("overwrite").parquet(output_path)
        masked_df.write.mode("overwrite").parquet(f"{gold_data_path}/{table_name}")
        print(f"Saved masked {table_name} data to {output_path}")
    except Exception as e:
        print(f"Error saving {table_name}: {e}")


Processing table: policy
+--------+------------+----------+----------+----------+--------------+--------+
|PolicyID|PolicyNumber|PolicyType| StartDate|   EndDate|PolicyHolderID|StatusID|
+--------+------------+----------+----------+----------+--------------+--------+
|       1|       P1001|      Auto|2023-01-01|2024-01-01|             1|       1|
|       2|       P1002|      Home|2023-02-01|2024-02-01|             2|       1|
|       3|       P1003|      Life|2023-03-01|2043-03-01|             3|       1|
|       4|       P1004|    Health|2023-04-01|2024-04-01|             4|       4|
|       5|       P1005|      Auto|2023-05-01|2024-05-01|             5|       1|
+--------+------------+----------+----------+----------+--------------+--------+
only showing top 5 rows
Saved masked policy data to wasbs://output-container@storage2madhu.blob.core.windows.net/raw/silver//policy
Processing table: policyholder
+--------------+---------+--------+-----------+---------------+------------+-------